# Podstawy OOP i testów w pytest — rozwiązania

Wersja na 4-godzinny dzień zajęć.

Notebook zawiera kompletne rozwiązania ćwiczeń dotyczących:

- klas i obiektów,
- `__init__` i `self`,
- atrybutów i metod,
- walidacji i wyjątków,
- testów w `pytest`,
- parametryzacji,
- fixtures,
- podstaw coverage.

# Opis danych

## Nazwa zbioru

**Knowledge & Support Application Domain Data**

## Źródło

Ćwiczenia nie korzystają z zewnętrznego datasetu. Dane reprezentują typowe encje biznesowe aplikacji backendowej do zarządzania dokumentami i zgłoszeniami użytkowników.

## Encje

### User

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| `email` | `str` | Adres e-mail użytkownika |
| `role` | `str` | Rola: `user`, `agent`, `admin` |
| `is_active` | `bool` | Stan aktywności konta |

### Document

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| `document_id` | `int` | Identyfikator dokumentu |
| `title` | `str` | Tytuł dokumentu |
| `content` | `str` | Treść dokumentu |
| `tags` | `list[str]` | Kategorie dokumentu |

### SupportTicket

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| `ticket_id` | `int` | Identyfikator zgłoszenia |
| `title` | `str` | Tytuł zgłoszenia |
| `author` | `User` | Autor zgłoszenia |
| `priority` | `str` | Priorytet: `low`, `medium`, `high` |
| `status` | `str` | Status zgłoszenia |
| `comments` | `list[str]` | Komentarze do zgłoszenia |

# Etap 1 — klasy i obiekty

## Zadanie 1. Pierwsza klasa `User`

**Rozwiązanie krok po kroku:**

1. Definiujemy klasę za pomocą `class`.
2. Pusta klasa może zawierać `pass`.
3. Tworzymy dwie niezależne instancje.
4. Sprawdzamy typ i relację z klasą.

In [ ]:
class User:
    pass


first_user = User()
second_user = User()

print(type(first_user))
print(isinstance(first_user, User))
print(first_user is second_user)

## Zadanie 2. Konstruktor i atrybuty

**Rozwiązanie krok po kroku:**

1. `__init__` przyjmuje dane potrzebne przy tworzeniu obiektu.
2. `self` wskazuje konkretną instancję.
3. Argument `role` ma wartość domyślną `user`.
4. Nowy użytkownik otrzymuje stan `is_active = True`.

In [ ]:
class User:
    def __init__(self, email: str, role: str = "user") -> None:
        self.email = email
        self.role = role
        self.is_active = True


regular_user = User(email="anna@example.com")
agent_user = User(
    email="agent@example.com",
    role="agent",
)

print(regular_user.email)
print(regular_user.role)
print(regular_user.is_active)

print(agent_user.email)
print(agent_user.role)
print(agent_user.is_active)

## Zadanie 3. Metody użytkownika

**Rozwiązanie krok po kroku:**

1. `deactivate` zmienia stan obiektu.
2. `can_manage_documents` zwraca wartość logiczną.
3. Uprawnienie zależy jednocześnie od roli i aktywności.

In [ ]:
class User:
    def __init__(self, email: str, role: str = "user") -> None:
        self.email = email
        self.role = role
        self.is_active = True

    def deactivate(self) -> None:
        self.is_active = False

    def can_manage_documents(self) -> bool:
        return (
            self.is_active
            and self.role in {"agent", "admin"}
        )


regular_user = User(email="user@example.com")
agent = User(email="agent@example.com", role="agent")
admin = User(email="admin@example.com", role="admin")

print(regular_user.can_manage_documents())
print(agent.can_manage_documents())
print(admin.can_manage_documents())

agent.deactivate()
print(agent.can_manage_documents())

## Zadanie 4. Walidacja użytkownika

**Rozwiązanie krok po kroku:**

1. Zbiór `ALLOWED_ROLES` przechowuje dozwolone role.
2. `strip()` pozwala wykryć pusty napis i same spacje.
3. Dla błędnych danych zgłaszamy `ValueError`.
4. Walidację wykonujemy przed zapisaniem atrybutów.

In [ ]:
class User:
    ALLOWED_ROLES = {"user", "agent", "admin"}

    def __init__(self, email: str, role: str = "user") -> None:
        if not email.strip():
            raise ValueError("Email cannot be empty")

        if role not in self.ALLOWED_ROLES:
            raise ValueError("Unsupported role")

        self.email = email
        self.role = role
        self.is_active = True

    def deactivate(self) -> None:
        self.is_active = False

    def can_manage_documents(self) -> bool:
        return self.is_active and self.role in {"agent", "admin"}


valid_user = User(email="anna@example.com")
print(valid_user.email)

try:
    User(email="   ")
except ValueError as error:
    print(error)

try:
    User(
        email="hero@example.com",
        role="superhero",
    )
except ValueError as error:
    print(error)

# Etap 2 — dokumenty

## Zadanie 5. Klasa `Document`

**Rozwiązanie krok po kroku:**

1. Konstruktor waliduje tytuł i treść.
2. Każdy dokument otrzymuje własną listę tagów.
3. Lista jest tworzona wewnątrz `__init__`.

In [ ]:
class Document:
    def __init__(
        self,
        document_id: int,
        title: str,
        content: str,
    ) -> None:
        if not title.strip():
            raise ValueError("Title cannot be empty")

        if not content.strip():
            raise ValueError("Content cannot be empty")

        self.document_id = document_id
        self.title = title
        self.content = content
        self.tags: list[str] = []


first_document = Document(
    document_id=1,
    title="Resetowanie hasła",
    content="Instrukcja resetowania hasła.",
)
second_document = Document(
    document_id=2,
    title="Konfiguracja konta",
    content="Instrukcja konfiguracji konta.",
)

first_document.tags.append("authentication")

print(first_document.tags)
print(second_document.tags)

## Zadanie 6. Metoda `add_tag`

**Rozwiązanie krok po kroku:**

1. Normalizujemy tag przez `strip().lower()`.
2. Odrzucamy pusty wynik.
3. Dodajemy tag tylko wtedy, gdy jeszcze go nie ma.

In [ ]:
class Document:
    def __init__(
        self,
        document_id: int,
        title: str,
        content: str,
    ) -> None:
        if not title.strip():
            raise ValueError("Title cannot be empty")

        if not content.strip():
            raise ValueError("Content cannot be empty")

        self.document_id = document_id
        self.title = title
        self.content = content
        self.tags: list[str] = []

    def add_tag(self, tag: str) -> None:
        normalized_tag = tag.strip().lower()

        if not normalized_tag:
            raise ValueError("Tag cannot be empty")

        if normalized_tag not in self.tags:
            self.tags.append(normalized_tag)


document = Document(
    document_id=1,
    title="Testowanie aplikacji",
    content="Wprowadzenie do pytest.",
)

document.add_tag("  Python  ")
document.add_tag("python")

print(document.tags)

# Etap 3 — testy użytkownika

Poniższą zawartość zapisz w pliku `tests/test_users.py`.

In [ ]:
test_users_py = '''
import pytest

from knowledge_app.users import User


def test_user_is_active_after_creation() -> None:
    # Arrange
    user = User(email="anna@example.com")

    # Act
    result = user.is_active

    # Assert
    assert result is True


def test_deactivate_changes_user_status() -> None:
    user = User(email="anna@example.com")

    user.deactivate()

    assert user.is_active is False


def test_user_rejects_empty_email() -> None:
    with pytest.raises(
        ValueError,
        match="Email cannot be empty",
    ):
        User(email="")


def test_user_rejects_unsupported_role() -> None:
    with pytest.raises(
        ValueError,
        match="Unsupported role",
    ):
        User(
            email="hero@example.com",
            role="superhero",
        )


@pytest.mark.parametrize(
    ("role", "expected"),
    [
        ("user", False),
        ("agent", True),
        ("admin", True),
    ],
)
def test_can_manage_documents_depends_on_role(
    role: str,
    expected: bool,
) -> None:
    user = User(
        email="person@example.com",
        role=role,
    )

    assert user.can_manage_documents() is expected


@pytest.mark.parametrize(
    "email",
    ["", " ", "   "],
    ids=["empty", "single-space", "multiple-spaces"],
)
def test_user_rejects_blank_email(email: str) -> None:
    with pytest.raises(ValueError):
        User(email=email)


def test_inactive_agent_cannot_manage_documents() -> None:
    user = User(
        email="agent@example.com",
        role="agent",
    )
    user.deactivate()

    assert user.can_manage_documents() is False
'''

print(test_users_py)

# Etap 4 — fixtures i testy dokumentu

Wspólne fixtures zapisz w `tests/conftest.py`.

In [ ]:
conftest_py = '''
import pytest

from knowledge_app.documents import Document
from knowledge_app.support import SupportTicket
from knowledge_app.users import User


@pytest.fixture
def regular_user() -> User:
    return User(email="user@example.com")


@pytest.fixture
def sample_document() -> Document:
    return Document(
        document_id=1,
        title="Resetowanie hasła",
        content="Instrukcja resetowania hasła.",
    )


@pytest.fixture
def support_ticket(
    regular_user: User,
) -> SupportTicket:
    return SupportTicket(
        ticket_id=1,
        title="Problem z logowaniem",
        author=regular_user,
    )
'''

print(conftest_py)

In [ ]:
test_documents_py = '''
import pytest

from knowledge_app.documents import Document


def test_document_starts_without_tags(
    sample_document: Document,
) -> None:
    assert sample_document.tags == []


def test_add_tag_changes_document(
    sample_document: Document,
) -> None:
    sample_document.add_tag("python")

    assert sample_document.tags == ["python"]


def test_fixture_returns_fresh_document(
    sample_document: Document,
) -> None:
    assert sample_document.tags == []


def test_add_tag_normalizes_value(
    sample_document: Document,
) -> None:
    sample_document.add_tag("  Python  ")

    assert sample_document.tags == ["python"]


def test_add_tag_does_not_add_duplicate(
    sample_document: Document,
) -> None:
    sample_document.add_tag("Python")
    sample_document.add_tag("python")

    assert sample_document.tags == ["python"]


def test_add_tag_rejects_blank_value(
    sample_document: Document,
) -> None:
    with pytest.raises(
        ValueError,
        match="Tag cannot be empty",
    ):
        sample_document.add_tag("   ")


def test_document_rejects_blank_title() -> None:
    with pytest.raises(
        ValueError,
        match="Title cannot be empty",
    ):
        Document(
            document_id=1,
            title="   ",
            content="Treść.",
        )


def test_document_rejects_blank_content() -> None:
    with pytest.raises(
        ValueError,
        match="Content cannot be empty",
    ):
        Document(
            document_id=1,
            title="Tytuł",
            content="   ",
        )
'''

print(test_documents_py)

# Etap 5 — mini-projekt `SupportTicket`

## Implementacja klasy

**Rozwiązanie krok po kroku:**

1. Konstruktor waliduje tytuł, autora i priorytet.
2. Status początkowy to `open`.
3. Każde zgłoszenie ma własną listę komentarzy.
4. `add_comment` normalizuje tekst.
5. `close` kontroluje zmianę stanu.

In [ ]:
class SupportTicket:
    ALLOWED_PRIORITIES = {"low", "medium", "high"}

    def __init__(
        self,
        ticket_id: int,
        title: str,
        author: User,
        priority: str = "medium",
    ) -> None:
        if not title.strip():
            raise ValueError("Title cannot be empty")

        if not author.is_active:
            raise ValueError("Author must be active")

        if priority not in self.ALLOWED_PRIORITIES:
            raise ValueError("Unsupported priority")

        self.ticket_id = ticket_id
        self.title = title
        self.author = author
        self.priority = priority
        self.status = "open"
        self.comments: list[str] = []

    def add_comment(self, comment: str) -> None:
        normalized_comment = comment.strip()

        if not normalized_comment:
            raise ValueError("Comment cannot be empty")

        self.comments.append(normalized_comment)

    def close(self) -> None:
        if self.status == "closed":
            raise ValueError("Ticket is already closed")

        self.status = "closed"


author = User(email="author@example.com")
ticket = SupportTicket(
    ticket_id=1,
    title="Problem z logowaniem",
    author=author,
)

ticket.add_comment("  Problem nadal występuje.  ")
ticket.close()

print(ticket.comments)
print(ticket.status)

## Testy `SupportTicket`

Poniższą zawartość zapisz w `tests/test_support.py`.

In [ ]:
test_support_py = '''
import pytest

from knowledge_app.support import SupportTicket
from knowledge_app.users import User


def test_ticket_has_expected_initial_state(
    support_ticket: SupportTicket,
) -> None:
    assert support_ticket.status == "open"
    assert support_ticket.priority == "medium"
    assert support_ticket.comments == []


def test_add_comment_adds_normalized_comment(
    support_ticket: SupportTicket,
) -> None:
    support_ticket.add_comment("  Problem nadal występuje.  ")

    assert support_ticket.comments == ["Problem nadal występuje."]


def test_add_comment_rejects_blank_comment(
    support_ticket: SupportTicket,
) -> None:
    with pytest.raises(
        ValueError,
        match="Comment cannot be empty",
    ):
        support_ticket.add_comment("   ")


def test_close_changes_ticket_status(
    support_ticket: SupportTicket,
) -> None:
    support_ticket.close()

    assert support_ticket.status == "closed"


def test_close_rejects_already_closed_ticket(
    support_ticket: SupportTicket,
) -> None:
    support_ticket.close()

    with pytest.raises(
        ValueError,
        match="Ticket is already closed",
    ):
        support_ticket.close()


@pytest.mark.parametrize(
    "priority",
    ["low", "medium", "high"],
)
def test_ticket_accepts_supported_priority(
    regular_user: User,
    priority: str,
) -> None:
    ticket = SupportTicket(
        ticket_id=1,
        title="Problem",
        author=regular_user,
        priority=priority,
    )

    assert ticket.priority == priority


@pytest.mark.parametrize(
    "priority",
    ["urgent", "critical", ""],
)
def test_ticket_rejects_unsupported_priority(
    regular_user: User,
    priority: str,
) -> None:
    with pytest.raises(
        ValueError,
        match="Unsupported priority",
    ):
        SupportTicket(
            ticket_id=1,
            title="Problem",
            author=regular_user,
            priority=priority,
        )


def test_ticket_rejects_blank_title(
    regular_user: User,
) -> None:
    with pytest.raises(
        ValueError,
        match="Title cannot be empty",
    ):
        SupportTicket(
            ticket_id=1,
            title="   ",
            author=regular_user,
        )


def test_ticket_rejects_inactive_author() -> None:
    user = User(email="user@example.com")
    user.deactivate()

    with pytest.raises(
        ValueError,
        match="Author must be active",
    ):
        SupportTicket(
            ticket_id=1,
            title="Problem",
            author=user,
        )
'''

print(test_support_py)

# Zadanie dodatkowe — proste dziedziczenie

## Klasa `AdminUser`

`AdminUser` jest szczególnym rodzajem `User`. Konstruktor klasy bazowej uruchamiamy za pomocą `super()`.

In [ ]:
class AdminUser(User):
    def __init__(self, email: str) -> None:
        super().__init__(
            email=email,
            role="admin",
        )


admin = AdminUser(email="admin@example.com")

print(isinstance(admin, User))
print(admin.role)
print(admin.can_manage_documents())

admin.deactivate()
print(admin.is_active)
print(admin.can_manage_documents())

In [ ]:
test_admin_py = '''
from knowledge_app.users import AdminUser, User


def test_admin_user_is_user_instance() -> None:
    admin = AdminUser(email="admin@example.com")

    assert isinstance(admin, User)


def test_admin_user_has_admin_role() -> None:
    admin = AdminUser(email="admin@example.com")

    assert admin.role == "admin"


def test_admin_user_inherits_deactivate() -> None:
    admin = AdminUser(email="admin@example.com")

    admin.deactivate()

    assert admin.is_active is False
'''

print(test_admin_py)

# Gotowy kod produkcyjny

## `src/knowledge_app/users.py`

In [ ]:
users_py = '''
class User:
    ALLOWED_ROLES = {"user", "agent", "admin"}

    def __init__(self, email: str, role: str = "user") -> None:
        if not email.strip():
            raise ValueError("Email cannot be empty")

        if role not in self.ALLOWED_ROLES:
            raise ValueError("Unsupported role")

        self.email = email
        self.role = role
        self.is_active = True

    def deactivate(self) -> None:
        self.is_active = False

    def can_manage_documents(self) -> bool:
        return self.is_active and self.role in {"agent", "admin"}


class AdminUser(User):
    def __init__(self, email: str) -> None:
        super().__init__(email=email, role="admin")
'''

print(users_py)

## `src/knowledge_app/documents.py`

In [ ]:
documents_py = '''
class Document:
    def __init__(
        self,
        document_id: int,
        title: str,
        content: str,
    ) -> None:
        if not title.strip():
            raise ValueError("Title cannot be empty")

        if not content.strip():
            raise ValueError("Content cannot be empty")

        self.document_id = document_id
        self.title = title
        self.content = content
        self.tags: list[str] = []

    def add_tag(self, tag: str) -> None:
        normalized_tag = tag.strip().lower()

        if not normalized_tag:
            raise ValueError("Tag cannot be empty")

        if normalized_tag not in self.tags:
            self.tags.append(normalized_tag)
'''

print(documents_py)

## `src/knowledge_app/support.py`

In [ ]:
support_py = '''
from knowledge_app.users import User


class SupportTicket:
    ALLOWED_PRIORITIES = {"low", "medium", "high"}

    def __init__(
        self,
        ticket_id: int,
        title: str,
        author: User,
        priority: str = "medium",
    ) -> None:
        if not title.strip():
            raise ValueError("Title cannot be empty")

        if not author.is_active:
            raise ValueError("Author must be active")

        if priority not in self.ALLOWED_PRIORITIES:
            raise ValueError("Unsupported priority")

        self.ticket_id = ticket_id
        self.title = title
        self.author = author
        self.priority = priority
        self.status = "open"
        self.comments: list[str] = []

    def add_comment(self, comment: str) -> None:
        normalized_comment = comment.strip()

        if not normalized_comment:
            raise ValueError("Comment cannot be empty")

        self.comments.append(normalized_comment)

    def close(self) -> None:
        if self.status == "closed":
            raise ValueError("Ticket is already closed")

        self.status = "closed"
'''

print(support_py)

# Konfiguracja pytest i coverage

Zawartość pliku `pyproject.toml`:

In [ ]:
pyproject_toml = '''
[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]
addopts = "-ra"

[tool.coverage.run]
source = ["knowledge_app"]
branch = true

[tool.coverage.report]
show_missing = true
skip_covered = true
'''

print(pyproject_toml)

# Coverage

Uruchom wszystkie testy:

    pytest -v

Raport z brakującymi liniami:

    pytest --cov=knowledge_app --cov-report=term-missing

Branch coverage:

    pytest --cov=knowledge_app --cov-branch --cov-report=term-missing

Raport HTML:

    pytest --cov=knowledge_app --cov-report=html

In [ ]:
coverage_review = {
    "users.py: unsupported role": (
        "Sprawdzenie tworzenia użytkownika z nieznaną rolą."
    ),
    "documents.py: blank tag": (
        "Sprawdzenie dodania pustego tagu."
    ),
    "support.py: inactive author": (
        "Sprawdzenie utworzenia zgłoszenia przez nieaktywnego autora."
    ),
}

for location, description in coverage_review.items():
    print(f"{location}: {description}")

# Przegląd z AI

In [ ]:
ai_test_review = [
    {
        "proposal": "Utworzenie zgłoszenia z pustym tytułem",
        "classification": "zgodna z wymaganiami",
        "justification": "Tytuł jest wymagany i podlega walidacji.",
    },
    {
        "proposal": "Utworzenie zgłoszenia przez nieaktywnego autora",
        "classification": "zgodna z wymaganiami",
        "justification": "Autor musi być aktywny.",
    },
    {
        "proposal": "Automatyczne zamknięcie po 30 dniach",
        "classification": "niezgodna z wymaganiami",
        "justification": "W wymaganiach nie ma operacji czasowych.",
    },
    {
        "proposal": "Ponowne otwarcie zgłoszenia",
        "classification": "wartościowa, ale poza zakresem",
        "justification": "Czterogodzinny wariant obejmuje tylko zamykanie.",
    },
]

for review in ai_test_review:
    print(review["classification"], "-", review["proposal"])

# Minimalny rezultat zajęć

Projekt powinien zawierać:

- klasę `User`,
- klasę `Document`,
- klasę `SupportTicket`,
- walidację i wyjątki,
- minimum 12 testów,
- minimum 3 testy wyjątków,
- minimum 1 test parametryzowany,
- minimum 1 fixture,
- raport coverage.

Polecenia końcowe:

    pytest -v
    pytest --cov=knowledge_app --cov-branch --cov-report=term-missing

# Checklista samooceny

- [x] Potrafię utworzyć klasę i instancję.
- [x] Potrafię używać `__init__` i `self`.
- [x] Potrafię napisać metodę zmieniającą stan.
- [x] Potrafię napisać metodę zwracającą wynik.
- [x] Potrafię walidować dane i zgłaszać wyjątki.
- [x] Potrafię napisać test według AAA.
- [x] Potrafię testować wyjątek.
- [x] Potrafię zastosować parametryzację.
- [x] Potrafię utworzyć fixture.
- [x] Rozumiem izolację testów.
- [x] Potrafię uruchomić raport coverage.
- [x] Rozumiem, że coverage nie gwarantuje jakości testów.